# Vídeo con YOLO: pose y movimiento corporal
Este notebook introduce la estimación de pose con un modelo YOLO de keypoints. A diferencia de la detección general de objetos, aquí no buscamos solo una caja, sino puntos anatómicos relevantes del cuerpo.

## Qué vamos a hacer
1. Cargar un vídeo o usar la cámara.
2. Aplicar un modelo YOLO de pose.
3. Visualizar keypoints y esqueleto corporal.
4. Calcular una medida simple, como el ángulo del codo, para conectar la pose con análisis de movimiento.

## Requisitos
- `ultralytics`
- `opencv-python`
- `numpy`
- entorno local si quieres usar ventanas con OpenCV

In [ ]:
# Instalación orientativa si hace falta
# !pip install -U ultralytics opencv-python

In [ ]:
from pathlib import Path
import os

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, Markdown

try:
    from ultralytics import YOLO
except Exception as e:
    print("Ultralytics no disponible:", e)

print("Directorio actual:", os.getcwd())

## Selección del vídeo

In [ ]:
VIDEO_PATH = Path("video.mp4")

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded)))
except Exception:
    pass

print(f"Vídeo seleccionado: {VIDEO_PATH}")
print(f"¿Existe?: {VIDEO_PATH.exists()}")

In [ ]:
if VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=True))

## Carga del modelo de pose

In [ ]:
pose_model = YOLO("yolov8n-pose.pt")
pose_model

## Pose sobre un frame de ejemplo

In [ ]:
def read_first_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el vídeo: {video_path}")
    ret, frame = cap.read()
    cap.release()
    if not ret or frame is None:
        raise ValueError("No se pudo leer el primer frame.")
    return frame

if VIDEO_PATH.exists():
    frame = read_first_frame(VIDEO_PATH)
    results = pose_model.predict(source=frame, verbose=False)
    annotated = results[0].plot()

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Estimación de pose sobre el primer frame")
    plt.show()

## Inferencia de pose sobre todo el vídeo

In [ ]:
# Visualización local de pose sobre vídeo
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = pose_model.predict(source=frame, verbose=False)[0]
    annotated = results.plot()
    cv2.imshow("YOLO Pose", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Acceso a keypoints numéricos
Además de dibujar el esqueleto, podemos acceder a las coordenadas de los puntos para hacer mediciones.

In [ ]:
if VIDEO_PATH.exists():
    frame = read_first_frame(VIDEO_PATH)
    results = pose_model.predict(source=frame, verbose=False)[0]

    if results.keypoints is not None and len(results.keypoints) > 0:
        kpts = results.keypoints.xy[0].cpu().numpy()
        print("Shape de keypoints:", kpts.shape)
        print(kpts[:5])
    else:
        print("No se detectaron personas con keypoints en ese frame.")

## Ejemplo: cálculo de ángulo en el codo
Un caso clásico consiste en medir el ángulo formado por hombro, codo y muñeca. Esto sirve para introducir biomecánica, gesto deportivo o análisis postural.

In [ ]:
def compute_angle(a, b, c):
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.degrees(np.arccos(cosine))

# En COCO, algunos índices habituales son:
# 5: hombro izquierdo, 7: codo izquierdo, 9: muñeca izquierda

if VIDEO_PATH.exists():
    frame = read_first_frame(VIDEO_PATH)
    results = pose_model.predict(source=frame, verbose=False)[0]

    if results.keypoints is not None and len(results.keypoints) > 0:
        kpts = results.keypoints.xy[0].cpu().numpy()
        shoulder = kpts[5]
        elbow = kpts[7]
        wrist = kpts[9]
        angle = compute_angle(shoulder, elbow, wrist)
        print(f"Ángulo aproximado del codo izquierdo: {angle:.2f} grados")
    else:
        print("No se pudo calcular el ángulo porque no hay keypoints válidos.")

## Cámara en vivo con pose

In [ ]:
# Pose en vivo con webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = pose_model.predict(source=frame, verbose=False)[0]
    annotated = results.plot()
    cv2.imshow("YOLO Pose en vivo", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Qué aporta la estimación de pose
- Permite analizar movimiento humano más allá de una caja de detección.
- Es útil para deporte, ergonomía, rehabilitación, interacción persona-máquina y vigilancia.
- Puede combinarse con series temporales para reconocer gestos o acciones.

In [ ]:
Markdown('''
**Actividad sugerida**

1. Prueba el modelo con un vídeo donde aparezca una persona de cuerpo completo.
2. Observa en qué situaciones los keypoints son más estables o más erráticos.
3. Calcula otro ángulo corporal, por ejemplo rodilla o hombro.
4. Como ampliación, representa la evolución del ángulo a lo largo de varios frames.
''')